# Transformación (Silver Layer)

## Importacion de recursos
En esta celda se cargan las librerías base (Pandas, NumPy) y el módulo centralizado utils.py. La centralización de rutas e importaciones garantiza que el pipeline sea reproducible y que no existan discrepancias en las rutas de acceso a los datos entre los distintos colaboradores del proyecto.

In [1]:
import utils
import pandas as pd
import numpy as np

## Leer extracted y transformar
En la capa Silver, el bloque unifica las fuentes fragmentadas y aplica reglas de limpieza y calidad. Se normalizan los nombres de las columnas para evitar errores de sintaxis en SQL y se imputan valores nulos (usando la mediana) para evitar sesgos en el entrenamiento del modelo. Este paso transforma el dato crudo en información confiable y estructurada.

In [2]:
# Carga de las 5 fuentes unificadas
conn_ext = utils.get_db_connection("extracted")
fuentes = [pd.read_sql(f"SELECT * FROM extracted_fuente_{i}", conn_ext) for i in range(1, 6)]
conn_ext.close()
df = pd.concat(fuentes, ignore_index=True)

# Limpieza de nombres y manejo de nulos
df.columns = [col.lower().strip() for col in df.columns]
for col in df.columns:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna("unknown")

# Persistencia fragmentada en Silver
conn_trans = utils.get_db_connection("transformed")
fragmentos_trans = np.array_split(df, 5)
for i, frag in enumerate(fragmentos_trans, 1):
    df_final = pd.DataFrame(frag, columns=df.columns)
    df_final.to_sql(f"transformed_fuente_{i}", conn_trans, if_exists="replace", index=False)
conn_trans.close()

## Generación del archivo
Se crea el transformation_log.json para documentar formalmente las acciones de limpieza realizadas. Registrar el volumen de registros procesados y las acciones específicas de calidad permite auditar la transformación y asegurar que el dataset final cumple con los requisitos técnicos para el análisis avanzado.

In [3]:
# Crear el log de transformación solicitado
log_data = {
    "etapa": "Silver - Movies",
    "registros_totales": len(df),
    "acciones": ["Imputación de nulos", "Normalización de cabeceras"],
    "fecha": utils.get_timestamp()
}
utils.save_log(utils.TRANSFORM_LOG_PATH, log_data)
print(f"✅ Capa Silver completada con {len(df)} registros.")

✅ Capa Silver completada con 100000 registros.
